# SEMMA — Cleveland Heart Disease Classification

This notebook demonstrates Sample, Explore, Modify, Model, and Assess. Upload `heart.csv` or `heart_disease.csv` to Colab, or place it in `data/raw/`. This is an educational prototype, not a diagnostic system.

In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report,roc_auc_score,average_precision_score,ConfusionMatrixDisplay
sns.set_theme(style='whitegrid'); RANDOM_STATE=42
candidates=list(Path('data/raw').glob('*.csv')) if Path('data/raw').exists() else []
candidates += [Path('/content/heart.csv'),Path('/content/heart_disease.csv')]
DATA_PATH=next((p for p in candidates if p.exists()),None)
if DATA_PATH is None: raise FileNotFoundError('Upload heart.csv or heart_disease.csv, or place it in data/raw/')
raw=pd.read_csv(DATA_PATH,na_values=['?']); print('Loaded',DATA_PATH,raw.shape); display(raw.head())

## S — Sample

We create a stratified holdout sample so the class proportions are preserved.

In [ ]:
df=raw.copy(); df.columns=[str(c).strip().lower().replace(' ','_') for c in df.columns]
target=next((c for c in ['target','condition','num','heartdisease','heart_disease'] if c in df.columns),df.columns[-1])
if target=='num': df[target]=(pd.to_numeric(df[target],errors='coerce')>0).astype('Int64')
else:
    vals=df[target].astype(str).str.lower().str.strip(); df[target]=vals.map({'yes':1,'true':1,'positive':1,'disease':1,'present':1,'1':1,'no':0,'false':0,'negative':0,'absent':0,'0':0}).fillna(pd.to_numeric(vals,errors='coerce'))
df=df.dropna(subset=[target]); y=df[target].astype(int); X=df.drop(columns=[target]);
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
print('Target:',target,'train:',X_train.shape,'test:',X_test.shape); display(y.value_counts(normalize=True).rename('proportion').to_frame())

## E — Explore

We inspect missingness, numeric distributions, and correlations with the outcome.

In [ ]:
profile=pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'unique':df.nunique()}); display(profile)
plt.figure(figsize=(6,4)); sns.countplot(x=y); plt.title('Heart-disease target distribution'); plt.tight_layout(); plt.show()
numeric=df.select_dtypes(include='number');
if numeric.shape[1]>1: plt.figure(figsize=(10,7)); sns.heatmap(numeric.corr(),cmap='coolwarm',center=0); plt.title('Numeric feature correlations'); plt.tight_layout(); plt.show()

## M — Modify

Numeric fields are median-imputed and scaled. Categorical fields are imputed and one-hot encoded inside a pipeline so test information does not leak into training.

In [ ]:
num=X_train.select_dtypes(include='number').columns.tolist(); cat=[c for c in X_train.columns if c not in num]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models={'logistic_regression':LogisticRegression(max_iter=1000,class_weight='balanced',random_state=RANDOM_STATE),'decision_tree':DecisionTreeClassifier(max_depth=5,class_weight='balanced',random_state=RANDOM_STATE),'random_forest':RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1),'svm':SVC(probability=True,class_weight='balanced',random_state=RANDOM_STATE)}
ARTIFACTS=Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)

## M — Model and A — Assess

We compare four classifiers and assess discrimination plus class-specific errors. Recall is important because false negatives represent patients with disease that the model fails to flag.

In [ ]:
rows=[]; fitted={}
for name,est in models.items():
    pipe=Pipeline([('preprocess',pre),('model',est)]); pipe.fit(X_train,y_train); p=pipe.predict_proba(X_test)[:,1]; pred=(p>=.5).astype(int); rep=classification_report(y_test,pred,output_dict=True,zero_division=0)
    rows.append({'model':name,'roc_auc':roc_auc_score(y_test,p),'pr_auc':average_precision_score(y_test,p),'precision':rep['1']['precision'],'recall':rep['1']['recall'],'f1':rep['1']['f1-score']}); fitted[name]=pipe
results=pd.DataFrame(rows).sort_values('roc_auc',ascending=False); results.to_csv(ARTIFACTS/'model_comparison.csv',index=False); display(results)
best_name=results.iloc[0]['model']; best=fitted[best_name]; pred=best.predict(X_test); display(pd.DataFrame(classification_report(y_test,pred,output_dict=True,zero_division=0)).T)
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues'); plt.title(f'Confusion matrix — {best_name}'); plt.tight_layout(); plt.savefig(ARTIFACTS/'confusion_matrix.png',dpi=180); plt.show(); joblib.dump(best,ARTIFACTS/'best_heart_disease_pipeline.joblib')
print('Saved artifacts to',ARTIFACTS)

## SEMMA conclusion

The final report should explain how sampling, exploration, modification, modeling, and assessment affected the result. This model is for coursework only and requires external validation, clinical review, and fairness analysis before any real-world use.